# 01 - Training + Rollout Validation

In [3]:
import sys
from pathlib import Path

SRC_DIR = (Path.cwd().resolve() / '..' / 'src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Hard-reload local package modules so notebook always sees latest edits.
for name in list(sys.modules.keys()):
    if name == 'course_project' or name.startswith('course_project.'):
        del sys.modules[name]

from course_project.config import ExperimentConfig
from course_project.runner import run_experiment


In [4]:
# Shared config for fair comparison
common_cfg = dict(
    train_dataset='../data/data_aux_opt_lowT_448sims_noang_bidirect_train5.pt',
    val_dataset='../data/data_aux_opt_lowT_448sims_noang_bidirect_test244.pt',
    output_root='../results',
    device='cuda',
    pos_dim=2,
    history=1,
    node_features='positions',
    limit=20,
    hidden_size=128,
    n_layers=2,
    learning_rate=1e-5,
    learning_rate_decay=0.996,
    epochs=80,
    val_every=10,
    rollout_steps=50,
    rollout_every=10,
    train_rollout_steps=1,
    train_rollout_loss_decay=0.9,
)

cfg_transformer_rollout = ExperimentConfig(
    run_name='transformer',
    model_type='spatial_transformer',
    model_extras={
        'num_mlp': 3,
        'K1': 128,
        'K2': 32,
        'transformer_layers': 2,
        'transformer_heads': 2,
        'transformer_dropout': 0.0,
        'edge_aggr': 'mean',
        'k2_hidden_size': 32,
        'use_local_skip': True,
    },
    **common_cfg,
)

cfg_spatial_baseline = ExperimentConfig(
    run_name='spatial',
    model_type='spatial',
    model_extras={'num_mlp': 3},
    **common_cfg,
)

cfg_hybrid = ExperimentConfig(
    run_name='hybrid',
    model_type='hybrid',
    model_extras={
        'num_mlp': 3,
        'K1': 450,
        'K2': 2,
        'transformer_layers': 1,
        'transformer_heads': 1,
        'transformer_dropout': 0.0,
        'k2_hidden_size': 16,
    },
    **common_cfg,
)

m_hybrid = run_experiment(cfg_hybrid)
m_spatial = run_experiment(cfg_spatial_baseline)
# m_transformer = run_experiment(cfg_transformer_rollout)

# import pandas as pd
# pd.DataFrame([m_transformer, m_spatial, m_hybrid])[[
#     'run_name',
#     'model_type',
#     'rollout_r2',
#     'rollout_pos_mse',
#     'best_rollout_epoch',
#     'best_rollout_r2',
# ]]


[run] hybrid model=hybrid device=cuda output=../results/hybrid
[run] training...
[train] autoregressive loss steps=1 decay=0.9
[ep  10/80] tr=0.964 va=1.12 lr=9.61e-06 roll=r2=0.152 p=0.825 mse=2.54e-05 (244/244) cv=skipped gabs=0.3 gmax=0.301 g_over_l=0.759 norm=fz n=(90,4392) t=32.1s
[ep  20/80] tr=0.941 va=1.13 lr=9.23e-06 roll=r2=0.145 p=0.793 mse=2.82e-05 (244/244) cv=skipped gabs=0.3 gmax=0.301 g_over_l=0.658 norm=fz n=(90,4392) t=75.7s
[ep  30/80] tr=0.93 va=1.14 lr=8.87e-06 roll=r2=0.0407 p=0.754 mse=3.07e-05 (244/244) cv=skipped gabs=0.3 gmax=0.301 g_over_l=0.615 norm=fz n=(90,4392) t=32.3s
[ep  40/80] tr=0.92 va=1.14 lr=8.52e-06 roll=r2=0.00827 p=0.729 mse=3.17e-05 (244/244) cv=skipped gabs=0.3 gmax=0.301 g_over_l=0.645 norm=fz n=(90,4392) t=32.3s
[ep  50/80] tr=0.91 va=1.14 lr=8.18e-06 roll=r2=0.0143 p=0.724 mse=3.19e-05 (244/244) cv=skipped gabs=0.3 gmax=0.302 g_over_l=0.666 norm=fz n=(90,4392) t=32.1s
[ep  60/80] tr=0.903 va=1.14 lr=7.86e-06 roll=r2=0.00739 p=0.718 mse=3.3